## Imports

In [35]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import (
    FixedThresholdClassifier,
    GroupShuffleSplit,
    GridSearchCV,
    GroupKFold,
)
from sklearn.pipeline import Pipeline

### Load full dataset

In [36]:
source = pd.read_json("python_dataset.jsonl", lines=True)

### Prepare Dataset

In [37]:
df = source

# df = df.head(10000)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df[["task_id", "code", "ai_generated", "source"]]

### Split df into training and test

In [42]:
# Split code into input X and ouput y
all_task_ids = df["task_id"]
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(df, groups=df["task_id"]):
    train_task_ids = all_task_ids.iloc[train_idx]
    train_X, train_y = X.iloc[train_idx], y.iloc[train_idx]
    test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]


### Build Pipeline

In [ ]:
# Init a Pipeline with 3 steps:
# 1. Count vectorizer to make a count vector for each row in the provided data (only the code)
# 2. TfidfTransformer takes count vectors from previous step and produces tf-idf vectors
# 3. Logreg as our model

pipeline = Pipeline(
    steps=[
        ("cv", CountVectorizer()),
        ("tfidf", TfidfTransformer()),
        ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipeline.fit(train_X, train_y)

param_grid = {"logreg__C": [0.1, 1, 10, 100]}

group_k_fold = GroupKFold(n_splits=5)

grid_search_cv = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    refit=True,
    cv=group_k_fold,
    n_jobs=-1,
)

grid_search_cv.fit(train_X, train_y, groups=train_task_ids)

### Train model and predict

In [ ]:
pred_proba = pipeline.predict_proba(test_X)

positive_class_probs = pred_proba[:, 1]

In [15]:
# Using testdata to optimize threshold, will change that later on!
fpr, tpr, threshold = roc_curve(test_y, positive_class_probs)

for f, t, thr in zip(fpr, tpr, threshold):
    print(f"Threshhold: {thr:.4f} -> FPR: {f:.4f} -> TPR: {t:.4f}")

Threshhold: inf -> FPR: 0.0000 -> TPR: 0.0000
Threshhold: 1.0000 -> FPR: 0.0000 -> TPR: 0.0002
Threshhold: 1.0000 -> FPR: 0.0000 -> TPR: 0.0003
Threshhold: 1.0000 -> FPR: 0.0005 -> TPR: 0.0010
Threshhold: 1.0000 -> FPR: 0.0005 -> TPR: 0.0017
Threshhold: 1.0000 -> FPR: 0.0005 -> TPR: 0.0035
Threshhold: 0.9997 -> FPR: 0.0005 -> TPR: 0.0075
Threshhold: 0.9996 -> FPR: 0.0005 -> TPR: 0.0095
Threshhold: 0.9988 -> FPR: 0.0005 -> TPR: 0.0190
Threshhold: 0.9988 -> FPR: 0.0005 -> TPR: 0.0193
Threshhold: 0.9977 -> FPR: 0.0005 -> TPR: 0.0342
Threshhold: 0.9977 -> FPR: 0.0010 -> TPR: 0.0342
Threshhold: 0.9971 -> FPR: 0.0010 -> TPR: 0.0387
Threshhold: 0.9971 -> FPR: 0.0015 -> TPR: 0.0387
Threshhold: 0.9964 -> FPR: 0.0015 -> TPR: 0.0473
Threshhold: 0.9964 -> FPR: 0.0020 -> TPR: 0.0473
Threshhold: 0.9963 -> FPR: 0.0020 -> TPR: 0.0495
Threshhold: 0.9963 -> FPR: 0.0020 -> TPR: 0.0500
Threshhold: 0.9948 -> FPR: 0.0020 -> TPR: 0.0680
Threshhold: 0.9948 -> FPR: 0.0025 -> TPR: 0.0680
Threshhold: 0.9946 -> F